# 28 run liana condition specific

**Purpose:** Run condition-specific LIANA analyses.

**Original analysis notebook:** `25_run_LIANA_condition_specific_HVG8000_r0_6_v2_fixed_use_raw_false.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 44 v2 — LIANA condition-specific cell-cell communication analysis (HVG8000 r0.6)

This notebook runs **condition-specific LIANA** on high-confidence cell types.

Main goal:
- infer ligand–receptor communication networks separately in GBM, LGG, ndGBM, and rGBM;
- summarize malignant ↔ microenvironment signaling axes;
- compare prioritized candidate LR interactions across conditions.

Important interpretation note:
This is **condition-stratified CCC inference**, not a formal replicate-aware differential communication model. Results should be interpreted as prioritized candidate interactions and compared with cell-type abundance, expression programs, and biological context.


In [ ]:
# Imports and configuration
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
H5AD_PATH = PROJECT_DIR / "data/processed/integrated/GBM_core_4datasets_final_annotation_malignant_states.h5ad"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURE_DIR = PROJECT_DIR / "figures/LIANA_condition_specific_HVG8000_r0_6"
METADATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

OUT_PREFIX = "LIANA_condition_specific_HVG8000_r0_6"

# Condition-specific settings
CONDITIONS_ORDER = ["GBM", "LGG", "ndGBM", "rGBM"]

# Minimum cells per cell type inside a condition to include that cell type in that condition-specific LIANA run.
# 50 is used to keep LGG usable while still avoiding very sparse groups.
MIN_CELLS_PER_CELLTYPE_CONDITION = 50

# These cell types are excluded from condition-specific main analysis due to low/unstable counts.
EXCLUDE_CELLTYPES_CONDITION = {
    "NonMalignant_B_plasma_like",
    "NonMalignant_rare_mast_cell_like_low_count",
}

# LIANA settings
RESOURCE_NAME = "consensus"
EXPR_PROP = 0.10
LIANA_GROUPBY_KEY = "liana_celltype_condition_main"

# Prioritization settings
RANK_THRESHOLD = 0.05
TOP_N_PER_CONDITION_IF_TOO_FEW = 1000
TOP_N_GLOBAL_EXPORT_PER_CONDITION = 500

# Optional runtime safeguard. By default FALSE because condition-specific subsets are smaller than global run.
DOWNSAMPLE_FOR_RUNTIME = False
MAX_CELLS_PER_CELLTYPE_CONDITION = 12000
RANDOM_SEED = 1337

print("H5AD:", H5AD_PATH)
print("metadata:", METADATA_DIR)
print("figures:", FIGURE_DIR)


In [ ]:
# Helper functions
def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")
    return path

def choose_first_existing_column(df, candidates, required=True, label="column"):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Could not find {label}. Candidates={candidates}. Available={list(df.columns)[:80]}")
    return None

def normalize_text(x):
    return str(x).strip().replace("/", "_").replace("-", "_").replace(" ", "_").lower()

def make_liana_celltype_from_row(row, malignant_state_col, annotation_col, final_compartment_col):
    ms = str(row.get(malignant_state_col, ""))
    ann = str(row.get(annotation_col, ""))
    comp = str(row.get(final_compartment_col, ""))
    text = normalize_text(" ".join([ms, ann, comp]))

    # Malignant states
    if "cycling" in text and "malignant" in text:
        return "Malignant_cycling_candidate"
    if "hypoxia" in text and "malignant" in text:
        return "Malignant_hypoxia_stress_malignant"
    if "opc" in text and "npc" in text and "malignant" in text:
        return "Malignant_OPC_NPC_like_malignant"
    if ("ac_like" in text or "astro" in text or "_ac_" in text) and "reactive" in text and "malignant" in text:
        return "Malignant_AC_like_reactive_malignant"
    if "npc" in text and ("neuronal" in text or "neural" in text or "neuron" in text) and "malignant" in text:
        return "Malignant_NPC_neuronal_like_malignant"

    # Non-malignant high-confidence lineages
    if "oligodendrocyte" in text or "myelinating" in text:
        return "NonMalignant_oligodendrocyte_myelinating_reference_like"
    if "endothelial" in text:
        return "NonMalignant_endothelial_reference_like"
    if "mural" in text or "pericyte" in text or "fibroblast" in text:
        return "NonMalignant_mural_pericyte_fibroblast_like"
    if "b_plasma" in text or "plasma" in text or "b_cell" in text:
        return "NonMalignant_B_plasma_like"
    if "mast" in text:
        return "NonMalignant_rare_mast_cell_like_low_count"

    return "Review_or_ambiguous_excluded"

def lineage_simplified(celltype):
    c = str(celltype)
    if c.startswith("Malignant_"):
        return "malignant_or_cycling"
    if c.startswith("NonMalignant_"):
        return "microenvironment"
    return "excluded_or_review"

def biological_group(celltype):
    c = str(celltype)
    low = c.lower()
    if c.startswith("Malignant_"):
        if "opc_npc" in low:
            return "malignant_OPC_NPC"
        if "cycling" in low:
            return "malignant_cycling"
        if "hypoxia" in low:
            return "malignant_hypoxia_stress"
        if "ac_like" in low:
            return "malignant_AC_reactive"
        if "npc_neuronal" in low:
            return "malignant_NPC_neuronal"
        return "malignant_other"
    if "endothelial" in low:
        return "endothelial"
    if "mural" in low or "pericyte" in low or "fibroblast" in low:
        return "mural_pericyte_fibroblast"
    if "oligodendrocyte" in low:
        return "oligodendrocyte_reference"
    if "b_plasma" in low:
        return "B_plasma"
    if "mast" in low:
        return "mast_low_count"
    return "excluded_or_review"

def axis_from_source_target(source, target):
    s = lineage_simplified(source)
    t = lineage_simplified(target)
    if s == "malignant_or_cycling" and t == "malignant_or_cycling":
        return "malignant_or_cycling_to_malignant_or_cycling"
    if s == "malignant_or_cycling" and t == "microenvironment":
        return "malignant_or_cycling_to_microenvironment"
    if s == "microenvironment" and t == "malignant_or_cycling":
        return "microenvironment_to_malignant_or_cycling"
    if s == "microenvironment" and t == "microenvironment":
        return "microenvironment_to_microenvironment"
    return "other_or_excluded"

def biological_axis_from_source_target(source, target):
    return f"{biological_group(source)}_to_{biological_group(target)}"

def get_counts_layer(adata):
    if "counts" in adata.layers:
        return adata.layers["counts"]
    # fallback if X is count-like
    return adata.X

def copy_counts_to_X_and_log_normalize(adata):
    X = get_counts_layer(adata)
    adata.X = X.copy() if sp.issparse(X) else np.array(X, copy=True)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    return adata

def sample_stratified_downsample_indices(obs, group_col, max_per_group, seed=1337):
    rng = np.random.default_rng(seed)
    keep = []
    for group, idx in obs.groupby(group_col).indices.items():
        idx = np.array(idx)
        if len(idx) > max_per_group:
            idx = rng.choice(idx, size=max_per_group, replace=False)
        keep.extend(idx.tolist())
    return np.array(sorted(keep), dtype=int)

def find_rank_col(df):
    candidates = ["primary_rank", "aggregate_rank", "rank_aggregate", "magnitude_rank", "specificity_rank"]
    for c in candidates:
        if c in df.columns:
            return c
    # fallback: any column with rank in name
    rank_cols = [c for c in df.columns if "rank" in c.lower()]
    return rank_cols[0] if rank_cols else None

def find_lr_cols(df):
    ligand_col = None
    receptor_col = None
    for c in ["ligand_complex", "ligand", "ligand_symbol"]:
        if c in df.columns:
            ligand_col = c
            break
    for c in ["receptor_complex", "receptor", "receptor_symbol"]:
        if c in df.columns:
            receptor_col = c
            break
    return ligand_col, receptor_col

def add_liana_summary_columns(df):
    df = df.copy()
    if "source" not in df.columns or "target" not in df.columns:
        raise KeyError(f"LIANA result must contain source and target columns. Columns={list(df.columns)}")
    ligand_col, receptor_col = find_lr_cols(df)
    if ligand_col is None or receptor_col is None:
        raise KeyError(f"Could not identify ligand/receptor columns. Columns={list(df.columns)}")

    df["ligand"] = df[ligand_col].astype(str)
    df["receptor"] = df[receptor_col].astype(str)
    df["lr_pair"] = df["ligand"] + "→" + df["receptor"]
    df["source_clean"] = df["source"].astype(str)
    df["target_clean"] = df["target"].astype(str)
    df["source_lineage"] = df["source_clean"].map(lineage_simplified)
    df["target_lineage"] = df["target_clean"].map(lineage_simplified)
    df["axis"] = [axis_from_source_target(s, t) for s, t in zip(df["source_clean"], df["target_clean"])]
    df["biological_axis"] = [biological_axis_from_source_target(s, t) for s, t in zip(df["source_clean"], df["target_clean"])]
    df["source_target"] = df["source_clean"] + " → " + df["target_clean"]
    df["interaction_key"] = (
        df["source_clean"] + "|" + df["target_clean"] + "|" + df["ligand"] + "|" + df["receptor"]
    )
    rank_col = find_rank_col(df)
    if rank_col:
        df["rank_for_sorting"] = pd.to_numeric(df[rank_col], errors="coerce")
    else:
        df["rank_for_sorting"] = np.nan
    return df

def summarize_source_target(df):
    if len(df) == 0:
        return pd.DataFrame()
    return (
        df.groupby(["condition", "source_clean", "target_clean", "source_target", "axis", "biological_axis"], observed=True)
        .agg(
            n_interactions=("lr_pair", "size"),
            n_unique_lr=("lr_pair", "nunique"),
            min_rank=("rank_for_sorting", "min"),
            median_rank=("rank_for_sorting", "median"),
            top_lr=("lr_pair", lambda x: ";".join(pd.Series(x).astype(str).head(5)))
        )
        .reset_index()
        .sort_values(["condition", "n_interactions", "min_rank"], ascending=[True, False, True])
    )

def summarize_axis(df, col="axis"):
    if len(df) == 0:
        return pd.DataFrame()
    return (
        df.groupby(["condition", col], observed=True)
        .agg(
            n_interactions=("lr_pair", "size"),
            n_unique_source_target=("source_target", "nunique"),
            n_unique_lr=("lr_pair", "nunique"),
            min_rank=("rank_for_sorting", "min"),
            median_rank=("rank_for_sorting", "median"),
        )
        .reset_index()
        .sort_values(["condition", "n_interactions"], ascending=[True, False])
    )

def summarize_lr(df):
    if len(df) == 0:
        return pd.DataFrame()
    return (
        df.groupby(["condition", "lr_pair", "ligand", "receptor"], observed=True)
        .agg(
            n_source_target_pairs=("source_target", "nunique"),
            n_interaction_rows=("lr_pair", "size"),
            min_rank=("rank_for_sorting", "min"),
            median_rank=("rank_for_sorting", "median"),
            top_source_target=("source_target", lambda x: ";".join(pd.Series(x).astype(str).head(5)))
        )
        .reset_index()
        .sort_values(["condition", "n_source_target_pairs", "min_rank"], ascending=[True, False, True])
    )


In [ ]:
# Read final annotated object and resolve columns
adata = sc.read_h5ad(H5AD_PATH)
print(adata)
obs = adata.obs.copy()

condition_col = choose_first_existing_column(obs, ["condition", "disease", "diagnosis"], label="condition column")
sample_col = choose_first_existing_column(obs, ["sample_id", "sample", "Sample"], label="sample column")
patient_col = choose_first_existing_column(obs, ["patient_id", "patient", "Patient"], required=False, label="patient column")
dataset_col = choose_first_existing_column(obs, ["dataset_id", "dataset", "core_dataset"], required=False, label="dataset column")
cluster_col = choose_first_existing_column(obs, ["leiden_scvi_main", "leiden_scvi_0.6", "leiden"], required=False, label="cluster column")
malignant_state_col = choose_first_existing_column(obs, ["malignant_state", "condition_malignant_state", "final_malignant_state"], label="malignant state column")
annotation_col = choose_first_existing_column(obs, ["final_annotation_refined", "annotation_marker_based_HVG8000_r0_6", "annotation"], label="annotation column")
final_compartment_col = choose_first_existing_column(obs, ["final_compartment_major", "final_malignancy_group_copykat_marker"], label="final compartment column")

resolved = pd.DataFrame([
    {"field": "condition", "column": condition_col},
    {"field": "sample", "column": sample_col},
    {"field": "patient", "column": patient_col},
    {"field": "dataset", "column": dataset_col},
    {"field": "cluster", "column": cluster_col},
    {"field": "malignant_state", "column": malignant_state_col},
    {"field": "annotation", "column": annotation_col},
    {"field": "final_compartment", "column": final_compartment_col},
])
write_tsv_replace(resolved, METADATA_DIR / f"{OUT_PREFIX}_resolved_columns.tsv")

readiness = pd.DataFrame([
    {"metric": "h5ad_path", "value": str(H5AD_PATH)},
    {"metric": "n_cells_total", "value": adata.n_obs},
    {"metric": "n_genes", "value": adata.n_vars},
    {"metric": "condition_col", "value": condition_col},
    {"metric": "sample_col", "value": sample_col},
    {"metric": "malignant_state_col", "value": malignant_state_col},
    {"metric": "annotation_col", "value": annotation_col},
    {"metric": "final_compartment_col", "value": final_compartment_col},
    {"metric": "counts_layer_available", "value": "counts" in adata.layers},
    {"metric": "resource_name", "value": RESOURCE_NAME},
    {"metric": "expr_prop", "value": EXPR_PROP},
    {"metric": "min_cells_per_celltype_condition", "value": MIN_CELLS_PER_CELLTYPE_CONDITION},
])
write_tsv_replace(readiness, METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv")

display(resolved)
display(readiness)


In [ ]:
# Build clean LIANA condition-specific cell type labels
obs["liana_celltype_raw"] = obs.apply(
    lambda r: make_liana_celltype_from_row(
        r, malignant_state_col=malignant_state_col, annotation_col=annotation_col,
        final_compartment_col=final_compartment_col
    ),
    axis=1
)

obs["liana_lineage_simplified"] = obs["liana_celltype_raw"].map(lineage_simplified)
obs["liana_biological_group"] = obs["liana_celltype_raw"].map(biological_group)

adata.obs["liana_celltype_raw"] = obs["liana_celltype_raw"].astype("category")
adata.obs["liana_lineage_simplified"] = obs["liana_lineage_simplified"].astype("category")
adata.obs["liana_biological_group"] = obs["liana_biological_group"].astype("category")

celltype_counts_all = (
    obs["liana_celltype_raw"].value_counts(dropna=False)
    .rename_axis("liana_celltype_raw")
    .reset_index(name="n_cells")
)
celltype_counts_all["include_in_condition_specific_main"] = ~celltype_counts_all["liana_celltype_raw"].isin(EXCLUDE_CELLTYPES_CONDITION) & ~celltype_counts_all["liana_celltype_raw"].str.startswith("Review")
celltype_counts_all["lineage_simplified"] = celltype_counts_all["liana_celltype_raw"].map(lineage_simplified)
celltype_counts_all["biological_group"] = celltype_counts_all["liana_celltype_raw"].map(biological_group)

write_tsv_replace(celltype_counts_all, METADATA_DIR / f"{OUT_PREFIX}_celltype_counts_all.tsv")
display(celltype_counts_all)


In [ ]:
# Condition × cell type inclusion decisions
condition_celltype_counts = (
    obs.groupby([condition_col, "liana_celltype_raw"], observed=True)
    .size()
    .reset_index(name="n_cells")
    .rename(columns={condition_col: "condition"})
)

condition_celltype_counts["excluded_by_label"] = (
    condition_celltype_counts["liana_celltype_raw"].isin(EXCLUDE_CELLTYPES_CONDITION)
    | condition_celltype_counts["liana_celltype_raw"].astype(str).str.startswith("Review")
    | condition_celltype_counts["liana_celltype_raw"].astype(str).str.contains("rare_mast", case=False, regex=False)
)
condition_celltype_counts["passes_min_cells"] = condition_celltype_counts["n_cells"] >= MIN_CELLS_PER_CELLTYPE_CONDITION
condition_celltype_counts["include_condition_specific"] = (
    (~condition_celltype_counts["excluded_by_label"]) & condition_celltype_counts["passes_min_cells"]
)
condition_celltype_counts["lineage_simplified"] = condition_celltype_counts["liana_celltype_raw"].map(lineage_simplified)
condition_celltype_counts["biological_group"] = condition_celltype_counts["liana_celltype_raw"].map(biological_group)
condition_celltype_counts = condition_celltype_counts.sort_values(["condition", "include_condition_specific", "n_cells"], ascending=[True, False, False])

write_tsv_replace(condition_celltype_counts, METADATA_DIR / f"{OUT_PREFIX}_condition_celltype_inclusion_decisions.tsv")
display(condition_celltype_counts.head(50))

# Summary of how many cell types/cells will be used per condition
condition_run_plan = (
    condition_celltype_counts[condition_celltype_counts["include_condition_specific"]]
    .groupby("condition", observed=True)
    .agg(
        n_celltypes=("liana_celltype_raw", "nunique"),
        n_cells_included=("n_cells", "sum"),
        included_celltypes=("liana_celltype_raw", lambda x: ";".join(map(str, x)))
    )
    .reset_index()
)
write_tsv_replace(condition_run_plan, METADATA_DIR / f"{OUT_PREFIX}_condition_run_plan.tsv")
display(condition_run_plan)


In [ ]:
# Import LIANA
try:
    import liana as li
    try:
        rank_aggregate_callable = li.mt.rank_aggregate
        print("[LIANA] using li.mt.rank_aggregate")
    except Exception:
        from liana.method import rank_aggregate as rank_aggregate_callable
        print("[LIANA] using liana.method.rank_aggregate")
except Exception as e:
    raise ImportError(
        "LIANA is not available in this Python environment. "
        "Install/use the same environment where the global LIANA notebook ran successfully. "
        f"Original error: {repr(e)}"
    )

def run_liana_rank_aggregate(adata_sub, groupby_key):
    # IMPORTANT:
    # The condition-specific subsets do not have .raw initialized.
    # LIANA defaults to use_raw=True in some versions, which raises:
    #   ValueError('.raw is not initialized!')
    # We explicitly use the current log-normalized .X matrix instead.
    kwargs = dict(
        groupby=groupby_key,
        resource_name=RESOURCE_NAME,
        expr_prop=EXPR_PROP,
        use_raw=False,
        layer=None,
        verbose=True,
    )
    try:
        rank_aggregate_callable(adata_sub, **kwargs)
    except TypeError as e:
        # Backward-compatible fallback for older LIANA versions that may not accept layer/use_raw.
        print("[LIANA fallback] retrying without layer after TypeError:", repr(e))
        kwargs.pop("layer", None)
        rank_aggregate_callable(adata_sub, **kwargs)

    if "liana_res" not in adata_sub.uns:
        raise KeyError(f"LIANA finished but adata.uns['liana_res'] was not found. adata.uns keys={list(adata_sub.uns.keys())}")
    return adata_sub.uns["liana_res"].copy()


In [ ]:
# Run LIANA per condition
all_results = []
top_results = []
prioritized_results = []
run_records = []

for cond in CONDITIONS_ORDER:
    print("\n" + "=" * 90)
    print(f"[CONDITION] {cond}")
    cond_decisions = condition_celltype_counts[
        (condition_celltype_counts["condition"].astype(str) == cond) &
        (condition_celltype_counts["include_condition_specific"])
    ].copy()

    included_celltypes = cond_decisions["liana_celltype_raw"].astype(str).tolist()
    if len(included_celltypes) < 2:
        msg = f"Skipped {cond}: fewer than 2 included cell types."
        print("[SKIP]", msg)
        run_records.append({
            "condition": cond, "status": "skipped", "reason": msg,
            "n_cells": 0, "n_celltypes": len(included_celltypes), "n_liana_rows": 0,
            "n_prioritized": 0,
        })
        continue

    mask = (obs[condition_col].astype(str).values == cond) & obs["liana_celltype_raw"].astype(str).isin(included_celltypes).values
    adata_cond = adata[mask].copy()
    adata_cond.obs[LIANA_GROUPBY_KEY] = adata_cond.obs["liana_celltype_raw"].astype(str).astype("category")

    if DOWNSAMPLE_FOR_RUNTIME:
        keep_idx = sample_stratified_downsample_indices(
            adata_cond.obs, LIANA_GROUPBY_KEY,
            max_per_group=MAX_CELLS_PER_CELLTYPE_CONDITION,
            seed=RANDOM_SEED
        )
        adata_cond = adata_cond[keep_idx].copy()
        print(f"[DOWNSAMPLED] {cond}: n_cells={adata_cond.n_obs}")

    used_counts = (
        adata_cond.obs[LIANA_GROUPBY_KEY].astype(str)
        .value_counts()
        .rename_axis("liana_celltype")
        .reset_index(name="n_cells")
    )
    used_counts.insert(0, "condition", cond)
    write_tsv_replace(used_counts, METADATA_DIR / f"{OUT_PREFIX}_{cond}_celltype_counts_used.tsv")

    print("[SUBSET]", adata_cond)
    print(used_counts)

    # Normalize counts for LIANA
    adata_cond = copy_counts_to_X_and_log_normalize(adata_cond)

    try:
        res = run_liana_rank_aggregate(adata_cond, LIANA_GROUPBY_KEY)
        res = add_liana_summary_columns(res)
        res.insert(0, "condition", cond)
        rank_col = find_rank_col(res)
        if rank_col is None:
            print("[WARNING] no rank column found; prioritization will use first rows.")
            res_sorted = res.copy()
            prioritized = res_sorted.head(TOP_N_PER_CONDITION_IF_TOO_FEW).copy()
        else:
            res_sorted = res.sort_values("rank_for_sorting", ascending=True).copy()
            prioritized = res_sorted[res_sorted["rank_for_sorting"] <= RANK_THRESHOLD].copy()
            if len(prioritized) == 0:
                prioritized = res_sorted.head(TOP_N_PER_CONDITION_IF_TOO_FEW).copy()
                prioritized["prioritization_rule"] = f"top_{TOP_N_PER_CONDITION_IF_TOO_FEW}_fallback"
            else:
                prioritized["prioritization_rule"] = f"{rank_col}<={RANK_THRESHOLD}"
        top = res_sorted.head(TOP_N_GLOBAL_EXPORT_PER_CONDITION).copy()

        all_results.append(res)
        top_results.append(top)
        prioritized_results.append(prioritized)

        run_records.append({
            "condition": cond,
            "status": "completed",
            "reason": "",
            "n_cells": adata_cond.n_obs,
            "n_genes": adata_cond.n_vars,
            "n_celltypes": adata_cond.obs[LIANA_GROUPBY_KEY].nunique(),
            "included_celltypes": ";".join(sorted(adata_cond.obs[LIANA_GROUPBY_KEY].astype(str).unique())),
            "n_liana_rows": len(res),
            "n_top_exported": len(top),
            "n_prioritized": len(prioritized),
            "rank_col": rank_col,
        })
        print(f"[DONE] {cond}: rows={len(res)} prioritized={len(prioritized)}")

    except Exception as e:
        print("[ERROR]", cond, repr(e))
        run_records.append({
            "condition": cond,
            "status": "error",
            "reason": repr(e),
            "n_cells": adata_cond.n_obs,
            "n_genes": adata_cond.n_vars,
            "n_celltypes": adata_cond.obs[LIANA_GROUPBY_KEY].nunique(),
            "included_celltypes": ";".join(sorted(adata_cond.obs[LIANA_GROUPBY_KEY].astype(str).unique())),
            "n_liana_rows": 0,
            "n_top_exported": 0,
            "n_prioritized": 0,
            "rank_col": "",
        })

run_summary = pd.DataFrame(run_records)
write_tsv_replace(run_summary, METADATA_DIR / f"{OUT_PREFIX}_run_summary.tsv")
display(run_summary)


In [ ]:
# Combine and save LIANA results
if len(all_results) == 0:
    raise RuntimeError("No condition-specific LIANA results were produced.")

all_res = pd.concat(all_results, ignore_index=True)
top_res = pd.concat(top_results, ignore_index=True) if top_results else pd.DataFrame()
prioritized_res = pd.concat(prioritized_results, ignore_index=True) if prioritized_results else pd.DataFrame()

# Save full results compressed; user usually does not need to upload this.
all_path = METADATA_DIR / f"{OUT_PREFIX}_all_results.tsv.gz"
all_res.to_csv(all_path, sep="\t", index=False, compression="gzip")
print(f"[SAVED] {all_path} shape={all_res.shape}")

write_tsv_replace(top_res, METADATA_DIR / f"{OUT_PREFIX}_top_interactions_per_condition.tsv")
write_tsv_replace(prioritized_res, METADATA_DIR / f"{OUT_PREFIX}_prioritized_interactions.tsv")

display(prioritized_res.head(20))


In [ ]:
# Summaries
source_target_summary = summarize_source_target(prioritized_res)
axis_summary = summarize_axis(prioritized_res, "axis")
biological_axis_summary = summarize_axis(prioritized_res, "biological_axis")
lr_summary = summarize_lr(prioritized_res)

write_tsv_replace(source_target_summary, METADATA_DIR / f"{OUT_PREFIX}_source_target_summary.tsv")
write_tsv_replace(axis_summary, METADATA_DIR / f"{OUT_PREFIX}_axis_summary.tsv")
write_tsv_replace(biological_axis_summary, METADATA_DIR / f"{OUT_PREFIX}_biological_axis_summary.tsv")
write_tsv_replace(lr_summary, METADATA_DIR / f"{OUT_PREFIX}_ligand_receptor_summary.tsv")

display(axis_summary)
display(biological_axis_summary.head(30))
display(lr_summary.head(30))


In [ ]:
# Cross-condition presence/comparison table for prioritized interactions
if len(prioritized_res) > 0:
    presence = (
        prioritized_res.groupby(["interaction_key", "source_clean", "target_clean", "lr_pair", "ligand", "receptor", "axis", "biological_axis"], observed=True)
        .agg(
            n_conditions=("condition", "nunique"),
            conditions=("condition", lambda x: ";".join(sorted(pd.Series(x).astype(str).unique()))),
            min_rank_all=("rank_for_sorting", "min"),
            median_rank_all=("rank_for_sorting", "median"),
        )
        .reset_index()
        .sort_values(["n_conditions", "min_rank_all"], ascending=[False, True])
    )

    # Add per-condition best rank columns
    rank_pivot = prioritized_res.pivot_table(
        index="interaction_key",
        columns="condition",
        values="rank_for_sorting",
        aggfunc="min"
    ).reset_index()
    rank_pivot.columns = [str(c) if c == "interaction_key" else f"rank_{c}" for c in rank_pivot.columns]
    presence = presence.merge(rank_pivot, on="interaction_key", how="left")

else:
    presence = pd.DataFrame()

write_tsv_replace(presence, METADATA_DIR / f"{OUT_PREFIX}_prioritized_interaction_condition_presence.tsv")
display(presence.head(30))


In [ ]:
# Focused malignant/microenvironment axes
focused_axes = [
    "malignant_or_cycling_to_microenvironment",
    "microenvironment_to_malignant_or_cycling",
    "malignant_or_cycling_to_malignant_or_cycling",
]
focused = prioritized_res[prioritized_res["axis"].isin(focused_axes)].copy()
focused = focused.sort_values(["condition", "axis", "rank_for_sorting"], ascending=[True, True, True])

# Candidate biologically interpretable LR families
keywords = [
    "PTN", "MDK", "PTPRZ1", "NCL", "SDC3",
    "TIMP1", "CD63",
    "LGALS1", "ITGB1",
    "MIF", "EGFR", "CD74",
    "VIM", "CD44",
    "VEGFA", "KDR", "FLT1", "ITGB1",
    "APP", "APOE", "LRP1",
    "TGFB", "COL", "FN1", "SPP1", "CXCL", "CCL",
]
pattern = "|".join(keywords)
focused["matches_candidate_keyword"] = focused["lr_pair"].str.contains(pattern, case=False, regex=True)

candidate_interactions = focused[focused["matches_candidate_keyword"]].copy()
write_tsv_replace(focused, METADATA_DIR / f"{OUT_PREFIX}_focused_malignant_microenvironment_interactions.tsv")
write_tsv_replace(candidate_interactions, METADATA_DIR / f"{OUT_PREFIX}_candidate_axis_interactions_annotated.tsv")

candidate_summary = (
    candidate_interactions.groupby(["condition", "axis", "lr_pair"], observed=True)
    .agg(
        n_source_target=("source_target", "nunique"),
        min_rank=("rank_for_sorting", "min"),
        top_source_target=("source_target", lambda x: ";".join(pd.Series(x).astype(str).head(5)))
    )
    .reset_index()
    .sort_values(["condition", "n_source_target", "min_rank"], ascending=[True, False, True])
)
write_tsv_replace(candidate_summary, METADATA_DIR / f"{OUT_PREFIX}_candidate_axis_interaction_summary.tsv")

display(candidate_summary.head(50))


In [ ]:
# Simple figures
figure_records = []

def savefig(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    figure_records.append({"figure": path.name, "path": str(path)})

# Axis summary heatmap: n interactions by condition
if len(axis_summary) > 0:
    mat = axis_summary.pivot_table(index="axis", columns="condition", values="n_interactions", fill_value=0)
    mat = mat.reindex(columns=[c for c in CONDITIONS_ORDER if c in mat.columns])
    plt.figure(figsize=(8, 4.5))
    plt.imshow(mat.values, aspect="auto")
    plt.yticks(range(mat.shape[0]), mat.index)
    plt.xticks(range(mat.shape[1]), mat.columns, rotation=45, ha="right")
    plt.colorbar(label="n prioritized LR rows")
    plt.title("LIANA condition-specific axis summary")
    savefig(FIGURE_DIR / f"{OUT_PREFIX}_axis_summary_heatmap.png")

# Top biological axes
if len(biological_axis_summary) > 0:
    top_bio_axes = (
        biological_axis_summary.groupby("biological_axis", observed=True)["n_interactions"].sum()
        .sort_values(ascending=False)
        .head(20)
        .index
    )
    mat2 = biological_axis_summary[biological_axis_summary["biological_axis"].isin(top_bio_axes)].pivot_table(
        index="biological_axis", columns="condition", values="n_interactions", fill_value=0
    )
    mat2 = mat2.loc[top_bio_axes]
    mat2 = mat2.reindex(columns=[c for c in CONDITIONS_ORDER if c in mat2.columns])
    plt.figure(figsize=(9, 7))
    plt.imshow(mat2.values, aspect="auto")
    plt.yticks(range(mat2.shape[0]), mat2.index)
    plt.xticks(range(mat2.shape[1]), mat2.columns, rotation=45, ha="right")
    plt.colorbar(label="n prioritized LR rows")
    plt.title("Top condition-specific biological LIANA axes")
    savefig(FIGURE_DIR / f"{OUT_PREFIX}_biological_axis_summary_heatmap.png")

# Candidate LR counts
if len(candidate_summary) > 0:
    top_lrs = (
        candidate_summary.groupby("lr_pair", observed=True)["n_source_target"].sum()
        .sort_values(ascending=False)
        .head(20)
        .index
    )
    mat3 = candidate_summary[candidate_summary["lr_pair"].isin(top_lrs)].pivot_table(
        index="lr_pair", columns="condition", values="n_source_target", fill_value=0
    )
    mat3 = mat3.loc[top_lrs]
    mat3 = mat3.reindex(columns=[c for c in CONDITIONS_ORDER if c in mat3.columns])
    plt.figure(figsize=(8, 7))
    plt.imshow(mat3.values, aspect="auto")
    plt.yticks(range(mat3.shape[0]), mat3.index)
    plt.xticks(range(mat3.shape[1]), mat3.columns, rotation=45, ha="right")
    plt.colorbar(label="n source-target pairs")
    plt.title("Candidate LR axes across conditions")
    savefig(FIGURE_DIR / f"{OUT_PREFIX}_candidate_LR_condition_heatmap.png")

figure_manifest = pd.DataFrame(figure_records)
write_tsv_replace(figure_manifest, METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv")
display(figure_manifest)


In [ ]:
# Output manifest
manifest_paths = [
    METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_resolved_columns.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_celltype_counts_all.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_condition_celltype_inclusion_decisions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_condition_run_plan.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_run_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_top_interactions_per_condition.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_prioritized_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_source_target_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_axis_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_biological_axis_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_ligand_receptor_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_prioritized_interaction_condition_presence.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_focused_malignant_microenvironment_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_candidate_axis_interactions_annotated.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_candidate_axis_interaction_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_all_results.tsv.gz",
]
# include per-condition cell count files if they exist
for cond in CONDITIONS_ORDER:
    p = METADATA_DIR / f"{OUT_PREFIX}_{cond}_celltype_counts_used.tsv"
    if p.exists():
        manifest_paths.append(p)

manifest = pd.DataFrame({
    "file": [p.name for p in manifest_paths],
    "path": [str(p) for p in manifest_paths],
    "exists": [Path(p).exists() for p in manifest_paths],
    "size_mb": [round(Path(p).stat().st_size / (1024**2), 3) if Path(p).exists() else np.nan for p in manifest_paths],
})
write_tsv_replace(manifest, METADATA_DIR / f"{OUT_PREFIX}_output_manifest.tsv")
display(manifest)
print("[DONE] Condition-specific LIANA notebook finished.")
